In [ ]:
import nltk
import re
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.wsd import lesk
from nltk.corpus import wordnet
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

## Multilingual E-Commerce Reviews

In [ ]:
reviews = [
    ("en", "The iPhone 15 camera is amazing but the battery life could be better."),
    ("en", "I love this Samsung Galaxy tablet. The display is stunning!"),
    ("es", "El teléfono móvil tiene una cámara excelente pero la batería es corta."),
    ("es", "Me encanta esta tablet Samsung. La pantalla es impresionante!"),
    ("fr", "L'ordinateur portable est rapide mais le prix est trop élevé."),
    ("fr", "J'adore ce téléphone. L'appareil photo est fantastique!")
]

## Language Identification & Preprocessing

In [ ]:
LANG_STOPWORDS = {
    "en": set(stopwords.words('english')),
    "es": set(stopwords.words('spanish')),
    "fr": set(stopwords.words('french'))
}

def preprocess(text, lang):
    tokens = word_tokenize(text.lower())
    sw = LANG_STOPWORDS.get(lang, set())
    tokens = [t for t in tokens if t.isalpha() and t not in sw and len(t) > 2]
    return tokens

def preprocess_for_tfidf(text, lang):
    tokens = preprocess(text, lang)
    return " ".join(tokens)

print("Preprocessed reviews:")
for lang, text in reviews:
    tokens = preprocess(text, lang)
    print(f"[{lang}] {tokens}")

## Text Representation — TF-IDF Vectorization

In [ ]:
clean_texts = [preprocess_for_tfidf(text, lang) for lang, text in reviews]

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(clean_texts)

feature_names = vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    columns=feature_names
)

print(f"TF-IDF matrix shape: {tfidf_df.shape}")
print(f"Vocabulary size: {len(feature_names)}")
print("\nTF-IDF Feature Matrix:")
print(tfidf_df.round(4).to_string())

## Word Sense Disambiguation with Multilingual WordNet

In [ ]:
def multilingual_lesk(sentence, target_word, lang):
    tokens = word_tokenize(sentence.lower())
    synset = lesk(tokens, target_word)
    if synset is None:
        for ss in wordnet.synsets(target_word, lang=lang):
            return ss.name(), ss.definition()
        return "UNKNOWN", "No sense found"
    return synset.name(), synset.definition()

ambiguous_cases = [
    ("en", "The iPhone 15 camera is amazing but the battery life could be better.", "battery"),
    ("es", "El teléfono móvil tiene una cámara excelente pero la batería es corta.", "batería"),
    ("fr", "L'ordinateur portable est rapide mais le prix est trop élevé.", "ordinateur"),
]

print("Word Sense Disambiguation Results:")
for lang, sentence, target in ambiguous_cases:
    name, definition = multilingual_lesk(sentence, target, lang)
    print(f"[{lang}] Target: {target} -> {name}")
    print(f"       Definition: {definition[:80]}...")
    print()

## Sentiment Analysis

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

SENTIMENT_LEXICON = {
    "es": {
        "encanta": 2.0, "impresionante": 2.0, "excelente": 1.5, "bueno": 1.0,
        "malo": -1.0, "corto": -0.5, "elevado": -0.5, "mal": -1.0
    },
    "fr": {
        "fantastique": 2.0, "impressionnant": 2.0, "excellent": 1.5, "bon": 1.0,
        "cher": -1.0, "élevé": -0.5, "lent": -1.0, "trop": -0.5
    }
}

def analyze_sentiment(text, lang):
    if lang == "en":
        scores = sia.polarity_scores(text)
        compound = scores["compound"]
        label = "Positive" if compound >= 0.05 else "Negative" if compound <= -0.05 else "Neutral"
        return compound, label
    else:
        tokens = word_tokenize(text.lower())
        score = sum(SENTIMENT_LEXICON.get(lang, {}).get(t, 0) for t in tokens)
        label = "Positive" if score > 0 else "Negative" if score < 0 else "Neutral"
        return float(score), label

## Arrange Results by Sentiment

In [ ]:
results = []
for lang, text in reviews:
    compound, label = analyze_sentiment(text, lang)
    results.append({
        "lang": lang,
        "review": text,
        "compound": compound,
        "label": label
    })

arranged = sorted(results, key=lambda x: x["compound"], reverse=True)

print(f"{'Rank':<5} {'Lang':<6} {'Label':<10} {'Compound':<10} {'Review'}")
print("=" * 90)
for i, r in enumerate(arranged, 1):
    print(
        f"{i:<5} "
        f"{r['lang']:<6} "
        f"{r['label']:<10} "
        f"{r['compound']:<10.4f} "
        f"{r['review'][:55]}..."
    )